# 🏥 HCR4 — Finetuning TrOCR sur Ordonnances Médicales

Ce notebook fine-tune le modèle **TrOCR** (Microsoft) sur votre dataset d'ordonnances manuscrites pour améliorer la reconnaissance de l'écriture des médecins.

---

**Avant de commencer :**
1. Allez dans **Exécution > Modifier le type d'exécution > GPU (T4)**
2. Uploadez le fichier `hcr4_dataset.zip` dans le panneau Fichiers (à gauche)

---

## Étape 1 — Installation des dépendances

In [ ]:
!pip install -q transformers[torch] datasets evaluate jiwer Pillow scikit-learn

## Étape 2 — Upload et décompression du dataset

**Option A** : Upload direct (exécutez la cellule ci-dessous, une fenêtre s'ouvrira pour choisir `hcr4_dataset.zip`)

**Option B** : Si vous avez déjà uploadé le fichier manuellement dans le panneau Fichiers, passez directement à la décompression.

In [ ]:
# Option A : Upload interactif
from google.colab import files
uploaded = files.upload()  # Sélectionnez hcr4_dataset.zip

In [ ]:
# Décompression du dataset
!unzip -o hcr4_dataset.zip -d .

# Vérification
import os
n_images = len(os.listdir('dataset/images'))
print(f"\n✅ Dataset décompressé : {n_images} images trouvées")
print(f"📄 Fichier CSV : {'dataset/annotations.csv' if os.path.exists('dataset/annotations.csv') else '❌ INTROUVABLE'}")

## Étape 3 — Vérification du GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"✅ GPU détecté : {gpu_name} ({gpu_mem:.1f} GB VRAM)")
else:
    print("⚠️ Aucun GPU détecté ! Allez dans Exécution > Modifier le type d'exécution > GPU")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"📌 Device utilisé : {device}")

## Étape 4 — Chargement du modèle TrOCR et du processeur

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

MODEL_NAME = "microsoft/trocr-base-handwritten"
MAX_TARGET_LENGTH = 256

print(f"📥 Chargement de {MODEL_NAME}...")
processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

# Configuration du décodeur
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.max_length = MAX_TARGET_LENGTH
model.config.early_stopping = True
model.config.no_repeat_ngram_size = 3
model.config.length_penalty = 2.0
model.config.num_beams = 4

model.to(device)
print(f"✅ Modèle chargé sur {device}")

## Étape 5 — Préparation du Dataset PyTorch

In [ ]:
import pandas as pd
from torch.utils.data import Dataset
from PIL import Image

SEPARATOR = " | "

class PrescriptionDataset(Dataset):
    """Dataset PyTorch pour ordonnances manuscrites HCR4."""

    def __init__(self, csv_file, root_dir, processor, max_target_length=256):
        raw_df = pd.read_csv(csv_file)
        self.df = raw_df.drop_duplicates(subset='file_name', keep='last').reset_index(drop=True)
        self.root_dir = root_dir
        self.processor = processor
        self.max_target_length = max_target_length

        # Vérification des images manquantes
        missing = [f for f in self.df['file_name'] if not os.path.isfile(os.path.join(root_dir, str(f)))]
        if missing:
            print(f"⚠️ {len(missing)} image(s) manquante(s) : {missing[:5]}")

    def __len__(self):
        return len(self.df)

    def _normalize_text(self, raw_text):
        lines = [l.strip() for l in str(raw_text).strip().splitlines() if l.strip()]
        return SEPARATOR.join(lines)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_name = str(row['file_name'])
        text = self._normalize_text(row['text'])

        image_path = os.path.join(self.root_dir, file_name)
        try:
            image = Image.open(image_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (384, 384))

        pixel_values = self.processor(image, return_tensors='pt').pixel_values

        labels = self.processor.tokenizer(
            text, padding='max_length',
            max_length=self.max_target_length, truncation=True
        ).input_ids
        labels = [l if l != self.processor.tokenizer.pad_token_id else -100 for l in labels]

        return {
            'pixel_values': pixel_values.squeeze(),
            'labels': torch.tensor(labels)
        }

print("✅ Classe PrescriptionDataset définie.")

In [ ]:
from sklearn.model_selection import train_test_split

CSV_FILE = 'dataset/annotations.csv'
IMAGES_DIR = 'dataset/images'

# Lecture et dédoublonnage
df = pd.read_csv(CSV_FILE)
df = df.drop_duplicates(subset='file_name', keep='last').reset_index(drop=True)
print(f"📊 Total d'images uniques : {len(df)}")

# Split 85/15
train_df, eval_df = train_test_split(df, test_size=0.15, random_state=42)
train_df.reset_index(drop=True).to_csv('dataset/train.csv', index=False)
eval_df.reset_index(drop=True).to_csv('dataset/eval.csv', index=False)

train_dataset = PrescriptionDataset('dataset/train.csv', IMAGES_DIR, processor, MAX_TARGET_LENGTH)
eval_dataset = PrescriptionDataset('dataset/eval.csv', IMAGES_DIR, processor, MAX_TARGET_LENGTH)

print(f"🏋️ Entraînement : {len(train_dataset)} images")
print(f"🧪 Évaluation   : {len(eval_dataset)} images")

## Étape 6 — Configuration et lancement de l'entraînement

In [ ]:
import evaluate

cer_metric = evaluate.load('cer')

def compute_cer(pred):
    """Calcule le Character Error Rate (CER)."""
    labels_ids = pred.label_ids
    pred_ids = pred.predictions

    # Remplacer -100 par pad_token_id
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)

    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {'cer': cer}

print("✅ Métrique CER prête.")

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, default_data_collator

# ============================
# HYPER-PARAMETRES
# Modifiez ces valeurs selon vos besoins
# ============================
NUM_EPOCHS = 10
BATCH_SIZE = 8          # T4 (16GB) supporte 8 facilement
LEARNING_RATE = 5e-5
GRADIENT_ACCUMULATION = 2  # Batch effectif = 8 * 2 = 16

training_args = Seq2SeqTrainingArguments(
    output_dir='./checkpoints',
    predict_with_generate=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    fp16=torch.cuda.is_available(),
    logging_steps=5,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,
    report_to='none',
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    processing_class=processor,
    args=training_args,
    compute_metrics=compute_cer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=default_data_collator,
)

print(f"🚀 Prêt ! {NUM_EPOCHS} époques, batch effectif = {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"   Estimation : ~5-10 min sur GPU T4")

In [ ]:
# ============================
# LANCEMENT DE L'ENTRAÎNEMENT
# ============================
print("🏋️ Entraînement en cours...\n")
trainer.train()
print("\n✅ Entraînement terminé !")

## Étape 7 — Sauvegarde du modèle fine-tuné

In [ ]:
FINAL_DIR = './trocr_finetuned_hcr4'

trainer.save_model(FINAL_DIR)
processor.save_pretrained(FINAL_DIR)

print(f"💾 Modèle sauvegardé dans : {FINAL_DIR}")

# Lister les fichiers sauvegardés
for f in os.listdir(FINAL_DIR):
    size = os.path.getsize(os.path.join(FINAL_DIR, f)) / 1e6
    print(f"   {f} ({size:.1f} MB)")

## Étape 8 — Test du modèle fine-tuné

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# Charger le modèle finetuné
finetuned_model = VisionEncoderDecoderModel.from_pretrained(FINAL_DIR).to(device)
finetuned_processor = TrOCRProcessor.from_pretrained(FINAL_DIR)

# Tester sur quelques images du jeu d'évaluation
test_df = pd.read_csv('dataset/eval.csv').drop_duplicates(subset='file_name', keep='last').head(5)

print("=" * 70)
print("TEST DU MODÈLE FINE-TUNÉ")
print("=" * 70)

fig, axes = plt.subplots(len(test_df), 1, figsize=(12, 4 * len(test_df)))
if len(test_df) == 1:
    axes = [axes]

for i, (_, row) in enumerate(test_df.iterrows()):
    img_path = os.path.join(IMAGES_DIR, str(row['file_name']))
    image = Image.open(img_path).convert('RGB')

    pixel_values = finetuned_processor(image, return_tensors='pt').pixel_values.to(device)
    generated_ids = finetuned_model.generate(pixel_values)
    prediction = finetuned_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

    # Texte attendu
    expected = ' | '.join([l.strip() for l in str(row['text']).strip().splitlines() if l.strip()])

    axes[i].imshow(image)
    axes[i].set_title(f"Prédit: {prediction}\nAttendu: {expected}", fontsize=10, loc='left')
    axes[i].axis('off')

    print(f"\n📸 Image : {row['file_name']}")
    print(f"   Attendu : {expected}")
    print(f"   Prédit  : {prediction}")

plt.tight_layout()
plt.show()

## Étape 9 — Télécharger le modèle sur votre machine

Le modèle sera compressé en ZIP puis téléchargé automatiquement. Vous devrez ensuite le décompresser dans votre projet HCR4.

In [ ]:
# Compression du modèle
!zip -r trocr_finetuned_hcr4.zip trocr_finetuned_hcr4/

# Téléchargement automatique
from google.colab import files
files.download('trocr_finetuned_hcr4.zip')

print("\n" + "=" * 60)
print("📦 Téléchargement lancé !")
print("")
print("Pour utiliser ce modèle dans HCR4 :")
print("  1. Décompressez le ZIP dans : backend/ai/finetuning/")
print("  2. Modifiez backend/config.py :")
print('     HTR_MODEL_NAME = "backend/ai/finetuning/trocr_finetuned_hcr4"')
print("  3. Relancez l'application : python run.py")
print("=" * 60)